# sklearn Pipelines

## Overview

A `Pipeline` chains preprocessing steps and a final estimator into a single object. This eliminates the most common source of data leakage in ML workflows and makes code reproducible, deployable, and cross-validatable in a single step.

**Why pipelines matter:**

| Without pipeline | With pipeline |
|---|---|
| Preprocessing fit on full data before CV | Preprocessing refit on each CV fold |
| Inconsistent train/test transforms | Guaranteed identical transforms |
| Deploy model + scaler separately | Deploy one object |
| Manual refit order on new data | `.predict()` handles everything |

**Core components:**
- `Pipeline`: linear chain of (name, transform) steps + final estimator
- `ColumnTransformer`: apply different transforms to different column subsets
- `FeatureUnion`: concatenate outputs of parallel transform paths

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
n = 400
# Mixed-type dataset: numeric + categorical + missing values
df = pd.DataFrame({
    'elevation':   rng.uniform(50, 400, n),
    'nitrate':     rng.gamma(2, 2, n),
    'phosphorus':  rng.gamma(1.5, 1.5, n),
    'catchment':   rng.choice(['North','South','East','West'], n),
    'season':      rng.choice(['Spring','Summer','Autumn','Winter'], n),
    'treatment':   rng.choice(['control','restored'], n),
})
# Introduce missing values
for col, frac in [('nitrate',0.08),('phosphorus',0.05),('catchment',0.04)]:
    mask = rng.random(n) < frac
    df.loc[mask, col] = np.nan
log_odds = (-1.5 + 0.003*df['elevation'].fillna(df['elevation'].mean())
            - 0.15*df['nitrate'].fillna(df['nitrate'].mean()))
y = (1/(1+np.exp(-log_odds)) > 0.5).astype(int)
print(f"Shape: {df.shape}, missing:\n{df.isnull().sum()}")
print(f"Class balance: {y.mean():.2f}")

---
## ColumnTransformer: Different Steps per Column Type

In [ ]:
num_cols = ['elevation','nitrate','phosphorus']
cat_cols = ['catchment','season','treatment']

# Numeric: impute median -> scale
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
# Categorical: impute mode -> one-hot encode
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])
# Full pipeline: preprocessor + classifier
pipe = Pipeline([
    ('prep', preprocessor),
    ('clf',  LogisticRegression(random_state=42, max_iter=500)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, df, y, cv=cv, scoring='roc_auc')
print(f"CV AUC: {scores.mean():.3f} +/- {scores.std():.3f}")
print("Pipeline fits preprocessor on train fold only -- no leakage")

---
## Swapping Estimators

In [ ]:
# Change the final estimator without touching preprocessing
classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=500),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
}
results = {}
for name, clf in classifiers.items():
    pipe_swap = Pipeline([('prep', preprocessor), ('clf', clf)])
    sc = cross_val_score(pipe_swap, df, y, cv=cv, scoring='roc_auc')
    results[name] = sc
    print(f"{name:25s}: AUC={sc.mean():.3f} +/- {sc.std():.3f}")

fig, ax = plt.subplots(figsize=(8,4))
ax.boxplot([results[k] for k in classifiers],
            labels=list(classifiers.keys()), patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('CV AUC'); ax.set_title('Pipeline: Classifier Comparison')
plt.tight_layout(); plt.show()

---
## GridSearchCV with Pipelines

In [ ]:
from sklearn.model_selection import GridSearchCV

# Access pipeline steps with __ notation
param_grid = {
    'clf__n_estimators':  [100, 200],
    'clf__max_depth':     [3, 5, None],
    'prep__num__imputer__strategy': ['mean', 'median'],
}
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('clf',  RandomForestClassifier(random_state=42)),
])
gs = GridSearchCV(pipe_rf, param_grid, cv=cv,
                  scoring='roc_auc', n_jobs=-1)
gs.fit(df, y)
print(f"Best params: {gs.best_params_}")
print(f"Best CV AUC: {gs.best_score_:.3f}")
print("\nDouble underscore __ accesses nested pipeline parameters:")
print("  'clf__n_estimators' -> pipeline step 'clf', param 'n_estimators'")
print("  'prep__num__imputer__strategy' -> prep -> num -> imputer -> strategy")

In [ ]:
# Saving and loading pipelines
import tempfile, os
try:
    import joblib
    pipe_final = Pipeline([('prep', preprocessor),
                            ('clf', RandomForestClassifier(**{
                                k.replace('clf__',''):v
                                for k,v in gs.best_params_.items()
                                if k.startswith('clf__')},
                                random_state=42))])
    pipe_final.fit(df, y)
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, 'pipeline.joblib')
        joblib.dump(pipe_final, path)
        pipe_loaded = joblib.load(path)
        size_kb = os.path.getsize(path) / 1024
    preds = pipe_loaded.predict_proba(df[:5])[:,1]
    print(f"Pipeline saved and reloaded ({size_kb:.1f} KB)")
    print(f"Predictions on first 5 rows: {preds.round(3)}")
    print("\njoblib.dump() serialises the full pipeline including preprocessor")
    print("One file -> complete reproducible inference object")
except ImportError:
    print("pip install joblib  (usually installed with sklearn)")

---

## Common Pitfalls

**1. Fitting the scaler on the full dataset before creating the pipeline**  
The single most common ML leakage mistake. `StandardScaler().fit(X)` computed before CV uses test-fold statistics to normalise training data. Always put the scaler inside the pipeline so it is fitted only on the training portion of each CV fold.

**2. Using `set_params()` with the wrong step name**  
Pipeline parameters use double-underscore notation: `clf__max_depth`, not `max_depth`. A typo silently creates a new attribute instead of setting the intended parameter, and sklearn may not raise an error. Always verify param names with `pipe.get_params().keys()`.

**3. Applying ColumnTransformer without specifying remainder**  
By default, `ColumnTransformer` drops any columns not listed in its transformers. If a column is accidentally omitted, it silently disappears. Set `remainder="passthrough"` to keep unlisted columns, or explicitly list every column.

**4. Forgetting `sparse_output=False` in OneHotEncoder inside a pipeline**  
Older sklearn versions return a sparse matrix from `OneHotEncoder`. If the next step does not accept sparse input, this causes a cryptic error at fit time. Set `sparse_output=False` (sklearn ≥ 1.2) or `sparse=False` (older) to ensure dense output.

**5. Saving only the model weights and not the full pipeline**  
If the scaler and encoder are saved separately from the model, inference requires manually applying each preprocessing step in the correct order — a common source of production bugs. Always serialise the full pipeline as a single object with `joblib.dump(pipeline, path)`.
---
*python_methods_library - Samantha McGarrigle*